In [ ]:
# ================================
# 08-hybrid-formal.ipynb
# Formalized Non-Overlapping Hybrid (NOH) IA³ + LoRA
# Combines Hindi & Telugu, fixes parameter overlaps, and uses dual-LR optimization
# ================================

# ------------------------------
# 1. Environment Setup
# ------------------------------
!pip uninstall -y torchao
!pip install -q peft --no-deps
!pip install -q trl --no-deps
!pip install -q accelerate

import torch, transformers, datasets, peft
import os, time, math, numpy as np, pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, IA3Config, TaskType, get_peft_model
from datasets import Dataset
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score

print("CUDA available:", torch.cuda.is_available())

# ------------------------------
# 2. Configuration
# ------------------------------
MODEL_NAME = "xlm-roberta-base"
NUM_LABELS = 3
MAX_LENGTH = 128
BATCH_SIZE = 16
SEEDS = [42, 123, 456]
BUDGETS = [100, 500, 2000, 20000]
LANGUAGES = ["hi", "te"]

DATA_ROOT = "/kaggle/input/notebooks/venkatkolluu/02-data-preprocessingv2/data/processed"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ------------------------------
# 3. Helper Functions
# ------------------------------
def load_base_model():
    return AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS).cuda()

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def build_loaders(language, budget):
    train_df = pd.read_parquet(f"{DATA_ROOT}/{language}/train_{budget}.parquet")
    valid_df = pd.read_parquet(f"{DATA_ROOT}/{language}/valid.parquet")

    def tokenize(batch):
        return tokenizer(batch["premise"], batch["hypothesis"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

    train_ds = Dataset.from_pandas(train_df).rename_column("label", "labels").map(tokenize, batched=True)
    valid_ds = Dataset.from_pandas(valid_df).rename_column("label", "labels").map(tokenize, batched=True)

    keep = ["input_ids", "attention_mask", "labels"]
    train_ds.set_format("torch", columns=keep)
    valid_ds.set_format("torch", columns=keep)

    return DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True), DataLoader(valid_ds, batch_size=32)

# ------------------------------
# 4. Hybrid Architecture (Non-Overlapping)
# ------------------------------
def build_hybrid_model():
    base = load_base_model()
    
    # IA3 strictly on scaling/feedforward
    ia3_config = IA3Config(
        task_type=TaskType.SEQ_CLS,
        target_modules=["key", "intermediate.dense", "output.dense"],
        feedforward_modules=["intermediate.dense", "output.dense"],
        modules_to_save=["classifier"]
    )
    model = get_peft_model(base, ia3_config, adapter_name="ia3")

    # LoRA strictly on attention weights
    lora_config = LoraConfig(
        r=8, lora_alpha=16, lora_dropout=0.1, bias="none",
        task_type=TaskType.SEQ_CLS,
        target_modules=["query", "value"],
        modules_to_save=[]
    )
    model.add_adapter("lora", lora_config)
    model.set_adapter(["ia3", "lora"])
    return model

# ------------------------------
# 5. Training Loop with Dual-LR Optimizer
# ------------------------------
def train_and_evaluate_hybrid(language, budget, seed):
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed); np.random.seed(seed)
    train_loader, valid_loader = build_loaders(language, budget)
    model = build_hybrid_model()
    
    # Dual-LR Optimizer
    lora_p, ia3_p, head_p = [], [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        if "lora_" in n: lora_p.append(p)
        elif "ia3_" in n: ia3_p.append(p)
        else: head_p.append(p)
        
    optimizer = torch.optim.AdamW([
        {"params": lora_p, "lr": 1e-4},
        {"params": ia3_p, "lr": 5e-3},
        {"params": head_p, "lr": 1e-4}
    ])

    epochs = 3 if budget == 20000 else (5 if budget in [1000, 2000] else 10)
    steps_per_epoch = math.ceil(budget / BATCH_SIZE)
    total_steps = steps_per_epoch * epochs

    model.train()
    start_time = time.perf_counter()
    for _ in range(epochs):
        for batch in train_loader:
            batch = {k: v.cuda() for k, v in batch.items()}
            loss = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], labels=batch["labels"]).loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
    train_time = time.perf_counter() - start_time

    # Evaluate
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in valid_loader:
            batch = {k: v.cuda() for k, v in batch.items()}
            outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
            preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
            labels.extend(batch["labels"].cpu().numpy())

    res = {
        "method": "hybrid_ia3_lora", "language": language, "budget": budget, "seed": seed,
        "accuracy": round(accuracy_score(labels, preds), 6),
        "macro_f1": round(f1_score(labels, preds, average="macro"), 6),
        "trainable_params": count_trainable_params(model),
        "training_time_sec": round(train_time, 2)
    }
    del model; torch.cuda.empty_cache()
    return res

# ------------------------------
# 6. Execute Matrix
# ------------------------------
results_file = "/kaggle/working/hybrid_experiment_results.csv"
results = []

for lang in LANGUAGES:
    for budget in BUDGETS:
        for seed in SEEDS:
            print(f"Training Hybrid | Lang: {lang} | Budget: {budget} | Seed: {seed}")
            try:
                res = train_and_evaluate_hybrid(lang, budget, seed)
                print(f"  -> Acc: {res['accuracy']:.4f} | F1: {res['macro_f1']:.4f}")
                pd.DataFrame([res]).to_csv(results_file, mode='a', header=not os.path.exists(results_file), index=False)
            except Exception as e:
                print(f"  -> ERROR: {e}")

print("Hybrid evaluation complete.")

